# TeaLeafDiseaseBD — revision_v1 reproduction and deployment notebook

Rebuilds the 70-label screener of §III-F from the released artifacts, trains the student, and exports the
ONNX file that goes on the phone. **This is a new notebook, not the original August file**: every step is
checked against what the original run saved, and stops if it does not match.

| Step | Check against | Stops if |
|---|---|---|
| 2 records | 13,084 images, 7 classes | count differs |
| 3 split | `_split.json` (9,155 / 3,929) and labels in `features_dinov1_crop.npz` | any label differs |
| 5 crops + 6 features | cached DINO features (cosine) | median < 0.99 → re-extracts instead |
| 7 probe | `probe_auc_per_rep` in `E_distillation_5rep.json` | AUC differs > 1e-3 (cached features) |
| 8 student | published 5-seed AUC spread 0.977–0.992 | outside the spread (warns) |
| 9 ONNX | PyTorch logits | max diff > 1e-3 |

Runtime on a T4: copy ~3 min, crop cache ~4 min, one student ~6–10 min. Set `REPS` in Cell 0.

In [ ]:
# CELL 0 — CONFIG
DRIVE      = '/content/drive/MyDrive'
TEA_DRIVE  = f'{DRIVE}/Tea Leaf Dataset'                              # one folder per class
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
FEATS      = f'{RES}/features_dinov1_crop.npz'
SPLIT      = f'{RES}/_split.json'
EJSON      = f'{RES}/E_distillation_5rep.json'
ESCORES    = f'{RES}/E_scores_5rep.npz'
DEPLOY     = f'{RES}/deploy'
LOCAL      = '/content/tea_leaf_local'
CACHE      = '/content/crop_cache'
SEED, N_SHOT, CROP_PAD, IMG = 42, 10, 0.06, 224
REPS       = [1]          # seed draws to train students for; [0,1,2,3,4] regenerates Table 11 WITH saved weights
EXPORT_REP = 1            # which trained student to export (1 = draw closest to the 5-seed mean)
EPOCHS, BS, LR = 30, 64, 1e-3


In [ ]:
# CELL 1 — setup and local copy of the dataset
from google.colab import drive; drive.mount('/content/drive')
!pip -q install onnx onnxruntime timm
import os, io, re, json, time, glob, shutil, hashlib, random, warnings
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torchvision.models as tvm, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import ndimage
from scipy.stats import spearmanr
import cv2
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
os.makedirs(DEPLOY, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png')

def _cp(t):
    s, d = t
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(s, d)
t0 = time.time()
tasks = [(os.path.join(r, f), os.path.join(LOCAL, os.path.relpath(r, TEA_DRIVE), f))
         for r, _, fs in os.walk(TEA_DRIVE) for f in fs if f.lower().endswith(IMG_EXT)]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('local copy: %d files in %.0fs' % (len(tasks), time.time() - t0))


In [ ]:
# CELL 2 — records in the original order: classes alphabetical, files sorted within class
classes = sorted(d for d in os.listdir(LOCAL) if os.path.isdir(os.path.join(LOCAL, d)))
HEALTHY = classes.index(next(c for c in classes if 'healthy' in c.lower()))
RECS = [dict(rel=f'{c}/{f}', path=os.path.join(LOCAL, c, f), y=k, d=int(k != HEALTHY))
        for k, c in enumerate(classes) for f in sorted(os.listdir(os.path.join(LOCAL, c)))
        if f.lower().endswith(IMG_EXT)]
print('classes:', classes, '| healthy index', HEALTHY)
print('records:', len(RECS))
assert len(classes) == 7 and len(RECS) == 13084, 'dataset differs from the published release'


In [ ]:
# CELL 3 — restore the group-aware split from _split.json, check labels against the cached features
SPL = json.load(open(SPLIT)); F = np.load(FEATS)
test_rel = ['/'.join(p.split('/')[-2:]) for p in SPL['test_paths']]
by_rel = {r['rel']: r for r in RECS}
missing = [p for p in test_rel if p not in by_rel]
assert not missing, f'{len(missing)} test files not found, e.g. {missing[:3]}'
TEST = [by_rel[p] for p in test_rel]                       # exact published test order
tset = set(test_rel)
TRAIN = [r for r in RECS if r['rel'] not in tset]          # record order (original convention)
ytr, dtr = np.array([r['y'] for r in TRAIN]), np.array([r['d'] for r in TRAIN])
yte, dte = np.array([r['y'] for r in TEST]), np.array([r['d'] for r in TEST])
print('train %d | test %d | test diseased %.4f' % (len(TRAIN), len(TEST), dte.mean()))
assert (len(TRAIN), len(TEST)) == (SPL['n_train'], SPL['n_test'])
assert (yte == F['yte']).all() and (dte == F['dte']).all(), 'test labels differ from cached features'
TRAIN_ORDER_OK = len(ytr) == len(F['ytr']) and (ytr == F['ytr']).all()
print('train label sequence matches cache:', TRAIN_ORDER_OK, '(order within class is checked in Cell 6)')


In [ ]:
# CELL 4 — segmentation and leaf crop
# Prefer the ORIGINAL modules from Drive (shape_confound.leaf_mask, feature_shape_control.crop_to_leaf);
# fall back to the paper's recipe: Otsu on HSV V|S, largest component, fill holes, bbox + 6% pad, 224x224.
import sys
for m in ('shape_confound.py', 'feature_shape_control.py'):
    hit = glob.glob(f'{STUDY}/**/{m}', recursive=True)
    if hit: shutil.copy(hit[0], f'/content/{m}')
sys.path.insert(0, '/content')
try:
    import shape_confound as _sc, feature_shape_control as _fsc
    _ORIG = True
except Exception as e:
    _ORIG = False; print('original modules not found (%s) -> fallback recipe' % e)

def _fb_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0); lab, n = ndimage.label(m)
    if n == 0: return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def _fb_crop(rgb, m, pad=CROP_PAD, size=IMG):
    ys, xs = np.where(m); H, W = m.shape
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0, x1, y1 = int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))
    return Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)

def leaf_crop(path):
    rgb = np.array(Image.open(path).convert('RGB'))
    if _ORIG:
        try:
            try: m = _sc.leaf_mask(rgb)
            except Exception: m = _sc.leaf_mask(Image.fromarray(rgb))
            im = _fsc.crop_to_leaf(rgb, np.asarray(m).astype(bool))
            im = im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im).astype(np.uint8))
            return im.convert('RGB').resize((IMG, IMG)) if im.size != (IMG, IMG) else im.convert('RGB')
        except Exception:
            pass
    return _fb_crop(rgb, _fb_mask(rgb))
print('crop function:', 'ORIGINAL modules' if _ORIG else 'fallback recipe')


In [ ]:
# CELL 5 — crop cache (JPEG q95, same as the revision_v1 run), parallel
def cache_path(rec):
    return os.path.join(CACHE, hashlib.md5(rec['rel'].encode()).hexdigest() + '.jpg')
def _build(rec):
    cp = cache_path(rec)
    if not os.path.exists(cp):
        try: leaf_crop(rec['path']).save(cp, quality=95)
        except Exception as e: return rec['rel'], str(e)
    return None
t0 = time.time()
with ProcessPoolExecutor(os.cpu_count()) as ex:
    fails = [f for f in ex.map(_build, RECS, chunksize=32) if f]
print('crop cache: %d images, %d failures, %.0fs' % (len(RECS), len(fails), time.time() - t0))
assert not fails, fails[:5]


In [ ]:
# CELL 6 — DINO ViT-S/16 features: use the cache only if our crops reproduce it
TO_T = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
dino = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(device).eval()
@torch.no_grad()
def feats(recs, bs=128):
    out = []
    for i in range(0, len(recs), bs):
        x = torch.stack([TO_T(Image.open(cache_path(r)).convert('RGB')) for r in recs[i:i + bs]]).to(device)
        out.append(torch.nn.functional.normalize(dino(x).float(), dim=1).cpu().numpy())
    return np.concatenate(out)
rng = np.random.default_rng(0)
it, ie = rng.choice(len(TRAIN), 64, replace=False), rng.choice(len(TEST), 64, replace=False)
ct = (feats([TRAIN[i] for i in it]) * F['Xtr'][it]).sum(1) if TRAIN_ORDER_OK else np.zeros(1)
ce = (feats([TEST[i] for i in ie]) * F['Xte'][ie]).sum(1)
print('cosine to cache: train median %.4f | test median %.4f' % (np.median(ct), np.median(ce)))
USE_CACHE = TRAIN_ORDER_OK and np.median(ct) >= 0.99 and np.median(ce) >= 0.99
if USE_CACHE:
    Xtr, Xte = F['Xtr'], F['Xte']; print('-> using cached features (exact reproduction of the published run)')
else:
    print('-> cache does not align with these crops: re-extracting all features (numbers will differ slightly)')
    Xtr, Xte = feats(TRAIN), feats(TEST)
    np.savez(f'{RES}/features_dinov1_crop_rebuilt.npz', Xtr=Xtr, ytr=ytr, dtr=dtr, Xte=Xte, yte=yte, dte=dte)
del dino; torch.cuda.empty_cache()


In [ ]:
# CELL 7 — 70-label probe (multinomial logistic, C=1.0) and the binary score; reproduce published AUCs
J = json.load(open(EJSON))
def draw(rep, n=N_SHOT):
    r = np.random.default_rng(SEED * 1000 + rep)
    return np.concatenate([r.choice(np.where(ytr == c)[0], n, replace=False) for c in range(len(classes))])
def fit_probe(X, y, idx):
    return LogisticRegression(C=1.0, max_iter=2000).fit(X[idx], y[idx])
def binary_score(probe, X):
    return 1.0 - probe.predict_proba(X)[:, HEALTHY]
pa = [roc_auc_score(dte, binary_score(fit_probe(Xtr, ytr, draw(r)), Xte)) for r in range(5)]
print('probe AUC per draw:', np.round(pa, 4)); print('published         :', np.round(J['probe_auc_per_rep'], 4))
if USE_CACHE:
    assert np.allclose(pa, J['probe_auc_per_rep'], atol=1e-3), 'probe did not reproduce: stop'
    print('-> exact reproduction')


In [ ]:
# CELL 8 — train the student(s): prevalence-quantile hard pseudo-labels, MobileNetV3-Small (Sec III-F)
class ArrDS(Dataset):
    def __init__(self, recs, labels, train):
        self.r, self.l, self.train = recs, labels, train
        self.aug = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                              T.RandomRotation(20), T.ColorJitter(.2, .2, .2)])
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        im = Image.open(cache_path(self.r[i])).convert('RGB')
        if self.train: im = self.aug(im)
        return TO_T(im), int(self.l[i])

def train_student(recs, labels, seed):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2); m = m.to(device)
    dl = DataLoader(ArrDS(recs, labels, True), batch_size=BS, shuffle=True, num_workers=4)
    opt = optim.Adam(m.parameters(), lr=LR); sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    lf = nn.CrossEntropyLoss()
    for ep in range(EPOCHS):
        m.train()
        for x, y in dl:
            opt.zero_grad(); lf(m(x.to(device)), y.to(device)).backward(); opt.step()
        sch.step()
    return m.eval()

@torch.no_grad()
def student_scores(m, recs):
    dl = DataLoader(ArrDS(recs, np.zeros(len(recs)), False), batch_size=128, num_workers=4)
    return np.concatenate([torch.softmax(m(x.to(device)), 1)[:, 1].cpu().numpy() for x, _ in dl])

E5 = np.load(ESCORES)['probe_distil']; pub = [roc_auc_score(dte, e) for e in E5]
STUDENTS, LOG = {}, []
for rep in REPS:
    idx = draw(rep); probe = fit_probe(Xtr, ytr, idx); s_tr = binary_score(probe, Xtr)
    pi = 1.0 - float(dtr.mean()); tau = float(np.quantile(s_tr, pi))
    pseudo = (s_tr > tau).astype(int); pseudo[idx] = dtr[idx]
    t0 = time.time(); m = train_student(TRAIN, pseudo, seed=1000 + rep); s = student_scores(m, TEST)
    a = roc_auc_score(dte, s); rho = spearmanr(s, E5[rep])[0]
    torch.save(m.state_dict(), f'{DEPLOY}/student_rep{rep}.pt'); np.save(f'{DEPLOY}/student_scores_rep{rep}.npy', s)
    STUDENTS[rep] = m
    LOG.append(dict(rep=rep, tau=tau, pseudo_acc=float((pseudo == dtr).mean()), auc=float(a),
                    published_auc=float(pub[rep]), spearman_vs_published=float(rho), minutes=(time.time() - t0) / 60))
    print('rep %d | tau %.4f | pseudo acc %.4f | AUC %.4f (published %.4f) | Spearman %.3f | %.1f min'
          % (rep, tau, LOG[-1]['pseudo_acc'], a, pub[rep], rho, LOG[-1]['minutes']))
lo, hi = min(pub), max(pub)
for L in LOG:
    L['within_published_spread'] = bool(lo - 0.005 <= L['auc'] <= hi + 0.005)
    if not L['within_published_spread']: print('WARNING rep %d outside the published spread %.4f-%.4f' % (L['rep'], lo, hi))
json.dump(LOG, open(f'{DEPLOY}/students.json', 'w'), indent=2)


In [ ]:
# CELL 9 — export ONNX (opset 13, dynamic batch) and verify against PyTorch
import onnxruntime as ort
m = STUDENTS[EXPORT_REP].to('cpu').eval()
onnx_path = f'{DEPLOY}/tea_screener.onnx'
import onnx, inspect
kw = dict(opset_version=13, input_names=['input'], output_names=['logits'],
          dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}}, do_constant_folding=True)
if 'dynamo' in inspect.signature(torch.onnx.export).parameters:
    kw['dynamo'] = False          # legacy exporter: honours opset 13 and keeps weights inside one file
torch.onnx.export(m, torch.randn(1, 3, IMG, IMG), onnx_path, **kw)
om = onnx.load(onnx_path, load_external_data=True)          # force a single self-contained file
onnx.save_model(om, onnx_path, save_as_external_data=False)
for extra in glob.glob(onnx_path + '.data'): os.remove(extra)
opset = [o.version for o in om.opset_import if o.domain in ('', 'ai.onnx')][0]
print('exported opset %d, %.2f MB, single file' % (opset, os.path.getsize(onnx_path) / 1e6))
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
dl = DataLoader(ArrDS(TEST, np.zeros(len(TEST)), False), batch_size=128, num_workers=4)
s_onnx, maxdiff = [], 0.0
with torch.no_grad():
    for x, _ in dl:
        z = sess.run(None, {'input': x.numpy()})[0]; maxdiff = max(maxdiff, float(np.abs(z - m(x).numpy()).max()))
        z = np.exp(z - z.max(1, keepdims=True)); s_onnx.append(z[:, 1] / z.sum(1))
s_onnx = np.concatenate(s_onnx); auc = roc_auc_score(dte, s_onnx)
flag = s_onnx >= 0.5
rep = dict(export_rep=EXPORT_REP, onnx=os.path.basename(onnx_path), opset=int(opset), dynamic_batch=True,
           size_mb=os.path.getsize(onnx_path) / 1e6, params_m=sum(p.numel() for p in m.parameters()) / 1e6,
           max_logit_diff_vs_torch=maxdiff, test_auc=float(auc), test_acc=float((flag == dte).mean()),
           sens=float(flag[dte == 1].mean()), spec=float((~flag)[dte == 0].mean()),
           published_5seed_auc=[float(p) for p in pub], used_cached_features=bool(USE_CACHE),
           crop_function='original modules' if _ORIG else 'fallback recipe',
           preprocessing='Otsu leaf crop + 6% pad, 224x224 (JPEG q95), ToTensor, ImageNet mean/std',
           output='logits[:,1] = diseased', sha256=hashlib.sha256(open(onnx_path, 'rb').read()).hexdigest())
json.dump(rep, open(f'{DEPLOY}/export_report.json', 'w'), indent=2)
np.save(f'{DEPLOY}/tea_screener_test_scores.npy', s_onnx)
print(json.dumps(rep, indent=2))
assert maxdiff < 1e-3, 'ONNX does not match PyTorch'
assert rep['size_mb'] > 5, 'weights not embedded in the .onnx file'
print('\nDEPLOY THIS FILE:', onnx_path if rep['test_auc'] >= min(pub) - 0.005 else 'NO — AUC below the published spread')


In [ ]:
# CELL 10 — desktop latency (indicative only; the paper reports the phone, see deploy/benchmark_onnx.py)
so = ort.SessionOptions(); so.intra_op_num_threads = 1
s1 = ort.InferenceSession(onnx_path, so, providers=['CPUExecutionProvider'])
x = np.random.default_rng(0).standard_normal((1, 3, IMG, IMG)).astype(np.float32)
for _ in range(15): s1.run(None, {'input': x})
lat = []
for _ in range(100):
    t = time.perf_counter(); s1.run(None, {'input': x}); lat.append((time.perf_counter() - t) * 1000)
print('Colab CPU, 1 thread: median %.2f ms, p95 %.2f ms' % (np.median(lat), np.percentile(lat, 95)))
print('Next: copy tea_screener.onnx + export_report.json to me, and run benchmark_onnx.py on the phone.')
